# Model Notebook: Content-Based Recommender

This notebook covers model-stage work end to end: loading processed features, training cosine KNN, single-song queries, profile-vector recommendation drift simulation, and artifact saving.

## Cell Block 1 - Configuration
Define all constants in one place (paths, feature list, model settings, and decay factor).

In [15]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = Path("..")
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "features.csv"
SCALER_PATH = PROJECT_ROOT / "data" / "processed" / "scaler.pkl"
MODEL_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "knn_model.pkl"

FEATURE_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "valence",
    "tempo",
]

META_COLS = ["track_id", "track_name", "artists", "track_genre", "popularity"]
N_NEIGHBORS = 10
DECAY = 0.85

print(f"Features path: {FEATURES_PATH}")
print(f"Scaler path:   {SCALER_PATH}")
print(f"Model path:    {MODEL_OUTPUT_PATH}")

Features path: ..\data\processed\features.csv
Scaler path:   ..\data\processed\scaler.pkl
Model path:    ..\data\processed\knn_model.pkl


## Cell Block 2 - Load Processed Data
Load `features.csv`, split into model matrix `X` (8 scaled columns) and metadata `df_meta` for result display only.

In [16]:
df_features = pd.read_csv(FEATURES_PATH)
scaler = joblib.load(SCALER_PATH)

missing_feature_cols = [c for c in FEATURE_COLS if c not in df_features.columns]
missing_meta_cols = [c for c in META_COLS if c not in df_features.columns]
if missing_feature_cols:
    raise ValueError(f"Missing feature columns: {missing_feature_cols}")
if missing_meta_cols:
    raise ValueError(f"Missing metadata columns: {missing_meta_cols}")

X = df_features[FEATURE_COLS].to_numpy(dtype=np.float64)
df_meta = df_features[META_COLS].copy()

print("Loaded processed dataframe shape:", df_features.shape)
print("Feature matrix shape:", X.shape)
print("Metadata dataframe shape:", df_meta.shape)
print("Scaler loaded type:", type(scaler).__name__)
df_meta.head(3)

Loaded processed dataframe shape: (89741, 15)
Feature matrix shape: (89741, 8)
Metadata dataframe shape: (89741, 5)
Scaler loaded type: StandardScaler


,track_id,track_name,artists,track_genre,popularity
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,acoustic,73
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,acoustic,55
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,acoustic,57


## Cell Block 3 - Fit KNN Model
We use `NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='brute')` and fit on `X`.

Why `algorithm='brute'` for cosine:
- Brute-force search computes exact cosine distances over all candidates.
- Tree-based methods (KD-Tree/Ball Tree) are mainly optimized for Euclidean-like spaces and usually do not perform well for cosine distance in high-dimensional settings.
- For this setup, brute force gives reliable exact neighbors under cosine metric.

In [17]:
knn_model = NearestNeighbors(
    n_neighbors=N_NEIGHBORS,
    metric="cosine",
    algorithm="brute",
)
knn_model.fit(X)
print("KNN model fitted successfully on X.")

KNN model fitted successfully on X.


## Cell Block 4 - Single Song Query
Define `get_recommendations(song_name, df_meta, X, model, n=10)` and test it with at least two songs.

In [18]:
def get_recommendations(song_name, df_meta, X, model, n=10):
    matches = df_meta.index[
        df_meta["track_name"].fillna("").str.lower() == str(song_name).strip().lower()
    ].tolist()
    if not matches:
        raise ValueError(f"Song not found: {song_name}")

    query_idx = matches[0]
    query_vector = X[query_idx].reshape(1, -1)

    k = min(len(df_meta), n + 1)
    distances, indices = model.kneighbors(query_vector, n_neighbors=k)

    rec_indices = [idx for idx in indices[0].tolist() if idx != query_idx][:n]

    rec_df = df_meta.loc[rec_indices].copy()
    rec_df.insert(0, "index", rec_indices)

    distance_map = {int(i): float(d) for i, d in zip(indices[0], distances[0])}
    rec_df["cosine_distance"] = rec_df["index"].map(distance_map)
    rec_df = rec_df.reset_index(drop=True)
    return rec_df

sample_songs = df_meta["track_name"].dropna().drop_duplicates().head(2).tolist()
print("Testing songs:", sample_songs)

for song in sample_songs:
    print(f"\nRecommendations for: {song}")
    test_recs = get_recommendations(song, df_meta, X, knn_model, n=N_NEIGHBORS)
    print(test_recs.head(10).to_string(index=False))

Testing songs: ['Comedy', 'Ghost - Acoustic']

Recommendations for: Comedy
 index               track_id                    track_name                    artists track_genre  popularity  cosine_distance
 20767 6dFOwtd9iBMERardJvsIxY                   She's Royal               Tarrus Riley   dancehall          58         0.025746
   850 6hDBkm6B8HF9B4oATW28YN                     Pop Virus                Gen Hoshino    acoustic          47         0.031416
 84001 54LGxQGf5LPXNGjtzDZ5IE So Good (feat. Ty Dolla $ign) Zara Larsson;Ty Dolla $ign     swedish          51         0.040732
 59860 0TA2eTf6ADQMuYLB0Ciy2V                          我們的歌                Leehom Wang    mandopop          43         0.049572
 56729 4E0GiSMs5E60MvX3pLW70H                    The Git Up              Mini Pop Kids        kids           8         0.050116
 27285 0837WsNtheChnvelltzlN0                    Give It Up                Horace Andy         dub          27         0.050748
 87902 6aBFjYfhZrgJtxiKaip1BH

## Cell Block 5 - User Profile Vector
Define profile aggregation with exponential decay where the oldest item gets `decay^(n-1)` and the newest gets `1.0`.

In [19]:
def compute_profile_vector(history: list[dict], decay: float) -> np.ndarray:
    if not history:
        raise ValueError("history must not be empty")

    n = len(history)
    weights = np.array([decay ** (n - 1 - i) for i in range(n)], dtype=np.float64)
    weight_sum = weights.sum()

    stacked = np.vstack([item["vector"] for item in history])
    profile = (stacked * weights[:, None]).sum(axis=0) / weight_sum
    return profile

demo_history = [
    {"vector": X[0]},
    {"vector": X[1]},
    {"vector": X[2]},
]
demo_weights = [DECAY ** (len(demo_history) - 1 - i) for i in range(len(demo_history))]
print("Decay weights for 3-song history (oldest -> newest):", demo_weights)
profile_demo = compute_profile_vector(demo_history, DECAY)
print("Profile vector length:", len(profile_demo))
print("Profile vector (first 8 values):", np.round(profile_demo[:8], 4))

Decay weights for 3-song history (oldest -> newest): [0.7224999999999999, 0.85, 1.0]
Profile vector length: 8
Profile vector (first 8 values): [-0.3581 -1.2104 -0.5505 -0.0037  0.2001 -0.5355 -0.509  -1.3975]


## Cell Block 6 - Session Simulation
Simulate Song A -> recommendations, pick Song B from results -> update profile, then Song C -> update profile again, printing top-5 recs each step.

In [20]:
def recommend_from_profile(profile_vector, X, df_meta, model, n=5, exclude_indices=None):
    if exclude_indices is None:
        exclude_indices = []

    k = min(len(df_meta), n + len(exclude_indices) + 10)
    distances, indices = model.kneighbors(profile_vector.reshape(1, -1), n_neighbors=k)

    selected = []
    selected_dist = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx in exclude_indices:
            continue
        selected.append(int(idx))
        selected_dist.append(float(dist))
        if len(selected) >= n:
            break

    rec_df = df_meta.loc[selected].copy()
    rec_df.insert(0, "index", selected)
    rec_df["cosine_distance"] = selected_dist
    return rec_df.reset_index(drop=True)

# Song A: start from the first available track
song_a_idx = 0
song_a_name = df_meta.loc[song_a_idx, "track_name"]
history = [{"vector": X[song_a_idx]}]
picked_indices = [song_a_idx]

print(f"Step 1 - Song A: {song_a_name}")
profile_step_1 = compute_profile_vector(history, DECAY)
print("Profile vector (first 8 values):", np.round(profile_step_1[:8], 4))
recs_step_1 = recommend_from_profile(profile_step_1, X, df_meta, knn_model, n=5, exclude_indices=picked_indices)
print(recs_step_1.to_string(index=False))

# Song B: choose the first recommendation from step 1
song_b_idx = int(recs_step_1.loc[0, "index"])
song_b_name = recs_step_1.loc[0, "track_name"]
history.append({"vector": X[song_b_idx]})
picked_indices.append(song_b_idx)

print(f"\nStep 2 - Song B: {song_b_name}")
profile_step_2 = compute_profile_vector(history, DECAY)
print("Profile vector (first 8 values):", np.round(profile_step_2[:8], 4))
recs_step_2 = recommend_from_profile(profile_step_2, X, df_meta, knn_model, n=5, exclude_indices=picked_indices)
print(recs_step_2.to_string(index=False))

# Song C: choose the first recommendation from step 2
song_c_idx = int(recs_step_2.loc[0, "index"])
song_c_name = recs_step_2.loc[0, "track_name"]
history.append({"vector": X[song_c_idx]})
picked_indices.append(song_c_idx)

print(f"\nStep 3 - Song C: {song_c_name}")
profile_step_3 = compute_profile_vector(history, DECAY)
print("Profile vector (first 8 values):", np.round(profile_step_3[:8], 4))
recs_step_3 = recommend_from_profile(profile_step_3, X, df_meta, knn_model, n=5, exclude_indices=picked_indices)
print(recs_step_3.to_string(index=False))

Step 1 - Song A: Comedy
Profile vector (first 8 values): [ 0.6443 -0.676   0.3357  0.4905 -0.8752 -0.5355  0.934  -1.1336]
 index               track_id                    track_name                    artists track_genre  popularity  cosine_distance
 20767 6dFOwtd9iBMERardJvsIxY                   She's Royal               Tarrus Riley   dancehall          58         0.025746
   850 6hDBkm6B8HF9B4oATW28YN                     Pop Virus                Gen Hoshino    acoustic          47         0.031416
 84001 54LGxQGf5LPXNGjtzDZ5IE So Good (feat. Ty Dolla $ign) Zara Larsson;Ty Dolla $ign     swedish          51         0.040732
 59860 0TA2eTf6ADQMuYLB0Ciy2V                          我們的歌                Leehom Wang    mandopop          43         0.049572
 56729 4E0GiSMs5E60MvX3pLW70H                    The Git Up              Mini Pop Kids        kids           8         0.050116

Step 2 - Song B: She's Royal
Profile vector (first 8 values): [ 0.8829 -0.6802  0.4184  0.333  -0.9133 -0.53

## Cell Block 7 - Save Model Artifact
Persist the fitted KNN model and validate artifact existence and size.

In [21]:
MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(knn_model, MODEL_OUTPUT_PATH)

exists = MODEL_OUTPUT_PATH.exists()
size_bytes = MODEL_OUTPUT_PATH.stat().st_size if exists else 0

print(f"Saved model to: {MODEL_OUTPUT_PATH}")
print(f"Exists: {exists}")
print(f"Size (bytes): {size_bytes}")

Saved model to: ..\data\processed\knn_model.pkl
Exists: True
Size (bytes): 5744006
